# CreditIQ: Credit Risk Modeling & Default Prediction
### End-to-End Artificial Neural Network (ANN) on LendingClub Dataset (2007–2018 Q4)

This notebook implements the complete data science pipeline for CreditIQ:
1. **Dataset Inspection & EDA**: Checking target distribution, missingness, and data leakage.
2. **Target Definition**: Filtering resolved loans (`Fully Paid` vs `Charged Off` / `Default`).
3. **Feature Selection & Engineering**: Building the 20 underwriting inputs.
4. **Out-of-Time Split**: Train (<= 2016), Validation (2017), Test (2018) to avoid macro leakage.
5. **Preprocessing Pipeline**: Imputation, scaling, and one-hot encoding.
6. **Deep ANN Architecture**: TensorFlow/Keras with Batch Normalization and Dropout.
7. **Evaluation & Underwriting Policy**: ROC-AUC, PR-AUC, and Cost-Minimizing Cutoff.

In [ ]:
import os
import sys
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# Add src to path
sys.path.append(os.path.abspath("../src"))
from config import DEFAULT_RAW_FILE, SAMPLE_RAW_FILE, TARGET_MAP, RAW_FEATURE_COLUMNS
from preprocess import engineer_features, CreditIQPreprocessor
from model import build_credit_risk_ann, compute_optimal_threshold

print("Libraries loaded successfully!")

## 1. Load Data & Preliminary Schema Verification

In [ ]:
# Determine file path: compressed raw dataset or sample
DATA_PATH = DEFAULT_RAW_FILE if os.path.exists(DEFAULT_RAW_FILE) else SAMPLE_RAW_FILE
print(f"Loading dataset from: {DATA_PATH}")

compression = "gzip" if DATA_PATH.endswith(".gz") else None
df_raw = pd.read_csv(DATA_PATH, compression=compression, nrows=50000, low_memory=False)

print(f"Loaded shape: {df_raw.shape}")
df_raw.head(3)

## 2. Target Variable Analysis & Exclusion of Active Loans
In accordance with credit risk best practices, we restrict the model to **resolved loans**:
- `Fully Paid` = `0` (Creditworthy / Non-Default)
- `Charged Off` / `Default` = `1` (Default)
- Exclude: `Current`, `In Grace Period`, `Late`

In [ ]:
print("Raw Loan Status distribution:")
print(df_raw["loan_status"].value_counts(dropna=False))

# Filter resolved loans
resolved_df = df_raw[df_raw["loan_status"].isin(TARGET_MAP.keys())].copy()
resolved_df["target"] = resolved_df["loan_status"].map(TARGET_MAP)

default_rate = resolved_df["target"].mean() * 100
print(f"\nResolved Loans: {len(resolved_df):,} | Baseline Default Rate: {default_rate:.2f}%")

## 3. Feature Engineering
Transforming string fields (`term`, `emp_length`, `revol_util`), calculating `credit_hist_months`, `monthly_income`, and average `fico_score`.

In [ ]:
features_df = engineer_features(resolved_df)
y = resolved_df["target"].values
features_df.head(3)

## 4. Chronological Out-of-Time Train/Validation/Test Split
Time-based splitting prevents data leakage and mimics real-world lending operations where models are trained on past cohorts and deployed on future applicants.

In [ ]:
years = sorted(features_df["issue_year"].dropna().unique())
print("Origination cohorts:", years)

if len(years) >= 3:
    train_mask = features_df["issue_year"].isin(years[:-2])
    val_mask = features_df["issue_year"].isin([years[-2]])
    test_mask = features_df["issue_year"].isin([years[-1]])
else:
    n = len(features_df)
    train_mask = np.zeros(n, dtype=bool)
    val_mask = np.zeros(n, dtype=bool)
    test_mask = np.zeros(n, dtype=bool)
    train_mask[:int(n*0.7)] = True
    val_mask[int(n*0.7):int(n*0.85)] = True
    test_mask[int(n*0.85):] = True

X_train_raw, y_train = features_df[train_mask], y[train_mask]
X_val_raw, y_val = features_df[val_mask], y[val_mask]
X_test_raw, y_test = features_df[test_mask], y[test_mask]

print(f"Train size: {len(X_train_raw):,} | Val size: {len(X_val_raw):,} | Test size: {len(X_test_raw):,}")

## 5. Fit Preprocessing Pipeline

In [ ]:
preprocessor = CreditIQPreprocessor()
X_train = preprocessor.fit_transform(X_train_raw)
X_val = preprocessor.transform(X_val_raw)
X_test = preprocessor.transform(X_test_raw)

print(f"Transformed input dimension: {X_train.shape[1]} features")

## 6. Build and Train Artificial Neural Network (ANN)
Using He-normal initialization, Batch Normalization, Dropout, and class balancing.

In [ ]:
import tensorflow as tf
from tensorflow import keras

# Handle class imbalance
neg_count = np.sum(y_train == 0)
pos_count = np.sum(y_train == 1)
class_weight = {
    0: (1 / neg_count) * (len(y_train) / 2.0),
    1: (1 / pos_count) * (len(y_train) / 2.0)
}

model = build_credit_risk_ann(input_dim=X_train.shape[1], learning_rate=0.001)
model.summary()

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_roc_auc", mode="max", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=25,
    batch_size=256,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)

## 7. Model Evaluation on Out-of-Time Test Set & Threshold Optimization

In [ ]:
test_metrics = model.evaluate(X_test, y_test, return_dict=True)
for k, v in test_metrics.items():
    print(f"Test {k.upper()}: {v:.4f}")

y_test_probs = model.predict(X_test).flatten()
best_threshold, threshold_stats = compute_optimal_threshold(y_test, y_test_probs, cost_fn=5.0, cost_fp=1.0)

print(f"\nOptimal Underwriting Cutoff: {best_threshold:.4f}")
print("Underwriting Performance at Optimal Cutoff:", threshold_stats)